# 04a. Graph Engineering ①: 층 지도, 작업 계약, 전달 계약, 공유 예산

앞선 `03a`, `03b`에서 순차·병렬·분기·루프의 네 가지 기본 흐름을 구현했습니다. 하지만 흐름만 연결했다고 해서 곧바로 실제 운영 환경에 투입할 수는 없습니다. 각 Worker가 어떤 형식으로 결과를 반환해야 하는지, 모델 호출을 어디서 멈출 것인지, 중간에 실패하면 어떻게 처리할 것인지, 환불 같은 실제 쓰기 작업을 언제 실행할지가 아직 코드로 통제되지 않았기 때문입니다.

`04` 단원은 두 권의 노트북(`04a`, `04b`)으로 나뉩니다. 이번 **`04a`** 노트북에서는 앞의 두 가지 핵심 과제, 즉 **Worker가 주고받는 계약(Contract)** 과 **호출을 제한하는 공유 예산(budget)** 을 그래프에 심습니다. 부분 실패 처리와 승인 게이트는 다음 **`04b`** 에서 이어집니다. 관통 사례는 계속해서 고객 요청 `T-1001`(*"결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요."*)을 사용합니다.

## 학습 목표

- 이 과정에서 정의하는 **Graph Engineering**의 의미를 이해하고, 각 설계 요소가 2~8절의 어느 작업에 대응하는지 파악합니다(2~4절은 `04a`, 5~8절은 `04b`에서 다룹니다).
- `Prompt`, `Context`, `Harness`, `Graph`, `Loop` 다섯 개 층이 각각 무엇을 설계하는지 이해하고 이 과정의 실제 코드 부품과 연결합니다.
- 단순한 역할 프롬프트와 명시적인 **작업 계약**의 차이를 실제 Worker 출력으로 비교합니다.
- Worker 사이의 **전달 계약**을 Pydantic 스키마로 작성하고, 스키마 형태 통과와 실제 근거(Evidence) 검증이 서로 다른 층의 검사임을 확인합니다.
- 서브그래프를 노드로 붙일 때 State 칸 이름이 어긋나면 오류 없이 결과가 사라지는 것을 재현하고, 입력과 출력을 명시해 경계를 세웁니다.
- 요약 전달과 구조화된 근거 전달에서 발생하는 정보 손실을 **동일한 Worker 실행 결과** 위에서 공정하게 비교합니다.
- 병렬 Worker들이 공유하는 **예산(budget)** 을 모델 외부의 단일 카운터로 집계하고, 한도 초과 시 즉시 중단하여 실패를 투명하게 드러냅니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 층 지도 | 3 |
| 2 | Prompt: 작업 계약 | 6 |
| 3 | Context: 전달 계약과 손실, 서브그래프의 경계 | 12 |
| 4 | Harness: 공유 예산 | 5 |
| 5 | 직접 해보기와 정리 | 4 |

## 호출 규모

이 노트북은 `Day03_new/.env`에 설정된 키로 실제 Claude 모델을 호출합니다. API 키 없이 동작하는 대체 경로는 없습니다. 모델을 호출하는 셀은 실행 직전에 `announce` 함수가 호출 규모를 한 줄로 고지하며, 비용은 본인 API 키로 청구됩니다. 그래프 구조 확인, 검증 로직, 예산 카운터처럼 모델의 추론이 필요 없는 부분은 일반 Python 함수로 실행하고 출력에 `"모델 호출 0회"`를 표시합니다. 다이어그램 셀과 앞선 실행 결과를 표로 정리하기만 하는 셀 역시 모델을 호출하지 않습니다.

| 구분 | 유료 셀 수 | 모델 호출(예고 상한) |
|---|---:|---:|
| 2절 역할 프롬프트 Worker, 작업 계약 Worker | 2 | 4 + 4 |
| 3절 T-1001 Worker 3개 실행, 같은 작성자에게 전달물 3종 주기 | 2 | 12 + 3 |
| 4절 예산 2회 Worker 실행 | 1 | 2 |
| 기본 실행 합계 | 5 | 예고 상한 25 |

`직접 해보기` 섹션의 유료 실습 셀(해보기 1)에서 스위치를 켜면 모델 호출이 최대 4회 추가됩니다. 실제 호출 횟수는 실행할 때마다 모델이 도구를 몇 번 선택하느냐에 따라 달라질 수 있습니다.

아래 코드는 실습에 필요한 모듈을 불러오고 API 키 상태와 관통 사례(`T-1001`)를 확인합니다.

In [ ]:
# 표준 라이브러리 및 동시성 도구 불러오기
import inspect
import json
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Deep Agents 및 Pydantic 검증 예외 불러오기
from deepagents import create_deep_agent
from pydantic import ValidationError

# 실습용 계약(Contract), 작업대(Desk), 평가, 원장, 추적(Trace), Worker 모듈 불러오기
from mas_lab.contracts import (FinalAnswer, WorkerResult, WorkerTask, join_results, next_action, plan_hash, trim_payload,
                               validate_result)
from mas_lab.desk import Desk
from mas_lab.evaluation import evaluate_row, load_gold
from mas_lab.hybrid import ROLE_GOALS, ROLES, build_hybrid
from mas_lab.ledger import check_refund
from mas_lab.runner import make_desk
from mas_lab.structures import OUTCOME_GUIDE, scope_line
from mas_lab.trace import ModelMeter, Trace, timeline
from mas_lab.view import show_table
from mas_lab.workers import ROLE_TOOLS, TASK_CONTRACT, allowlist_middleware, contract_prompt, make_worker, run_worker
from rag_lab.evidence import Budget, BudgetExceeded
from rag_lab.settings import announce, key_status, make_model, require_key

# API 키 확인 및 모델 객체 생성 (모델 객체 생성 자체는 API를 호출하지 않습니다)
require_key()  # 키가 설정되어 있지 않으면 여기서 한국어 안내와 함께 멈춥니다.
print(key_status())
model = make_model()

# 사례 데이터 로드 및 사후 자동 검사용 기준 데이터(gold) 준비
cases = [json.loads(line) for line in Path("data/cases.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
case = next(c for c in cases if c["id"] == "complex-double-pay")
gold = load_gold()  # 사후 자동 검사에만 사용하며, 도구·프롬프트·게이트는 이 값을 읽지 않습니다.

print("관통 사례:", case["id"], "|", case["question"])
print("측정 장치:", Desk.__name__, Trace.__name__, ModelMeter.__name__)

## 1. 층 지도: 새로운 이름은 이전 기술을 없애지 않습니다

**문제.** 에이전트가 틀린 답변을 내놓았을 때 가장 흔한 첫 반응은 *"프롬프트를 고쳐 보자"* 입니다. 하지만 실제 원인이 프롬프트에 있지 않은 경우가 많습니다. 필요한 약관 조항이 하위 Worker에게 애초에 전달되지 않았을 수도 있고, 재시도 루프가 공유 예산을 모두 소진했을 수도 있으며, 실패한 Worker의 결과가 합류 과정에서 조용히 누락되었을 수도 있습니다. 정확한 해결책을 찾으려면 문제의 원인을 **다섯 개 층(Layer)** 으로 나누어 진단해야 합니다.

이 노트북에서는 에이전트 엔지니어링의 다섯 층을 다음 표와 같이 정의합니다. `Prompt`, `Context`, `Harness` 같은 용어들은 시기마다 유행처럼 등장했지만, 새로운 개념이 나왔다고 해서 이전 층의 중요성이 사라지는 것은 아닙니다. 정교한 Graph를 설계하더라도 명확하게 작성된 Prompt는 여전히 필수적입니다.

| 층 | 설계 대상 | 각 층에서 던져야 할 핵심 질문 |
|---|---|---|
| Prompt | 작업 지시 | 이 작업에서 무엇을 어떤 조건과 형식으로 수행하라고 지시했는가? |
| Context | 제공 정보 | 이번 호출에 어떤 정보를 넣고 무엇을 뺐는가? 빠진 정보는 어디에 기록되는가? |
| Harness | 실행 환경과 통제 | 어떤 도구를 허용하는가? 예산(budget) 한도는 얼마인가? 누가 검증하고 누가 중단시키는가? |
| Graph | 작업·상태·의존관계 | 어떤 작업이 어떤 작업을 기다리는가? 수집된 상태는 어디에 어떻게 병합되는가? |
| Loop | 반복·재시도·종료 | 어떤 조건에서 재시도하고, 언제 사람에게 넘기며, 언제 최종 종료하는가? |

**이 과정의 Graph Engineering 정의.** 이 과정에서 말하는 **Graph Engineering**이란 작업을 노드(Node)와 간선(Edge)으로 명시하고, 각 노드마다 **누가 판단하는지**(코드·모델·사람), **무엇을 주고받는지**(계약), **언제 멈추고 다시 시도하는지**(예산·실패별 분기·승인 게이트)를 그래프 코드에 명확히 새겨 넣는 작업입니다. 즉, 다섯 층 지도의 `Graph` 층을 뼈대로 삼아 나머지 네 층(`Prompt`·`Context`·`Harness`·`Loop`)의 부품들을 적절한 노드와 간선에 배치하는 일입니다. `04a`의 2~4절과 `04b`의 5~8절에서 이 부품들을 층별로 하나씩 구현하여 결합합니다.

아래 코드에서는 이 과정에서 실제로 사용하는 핵심 함수와 클래스들을 다섯 층에 대응시켜 표로 출력합니다. 각 부품이 정의된 파일 경로와 줄 번호는 Python의 `inspect` 모듈로 직접 읽어 옵니다. 표의 마지막 열(`없으면 생기는 실패`)을 읽으며 각 부품이 빠졌을 때 어떤 문제가 발생할지 먼저 떠올려 보세요.

In [ ]:
# inspect 모듈을 사용해 각 부품이 정의된 실제 파일 경로와 줄 번호를 확인합니다.
def where(obj) -> str:
    """객체가 정의된 소스 파일의 상대 경로와 시작 줄 번호를 반환합니다."""
    file = Path(inspect.getsourcefile(obj)).resolve().relative_to(Path.cwd().resolve())
    return f"{file}:{inspect.getsourcelines(obj)[1]}"

# 다섯 개 층에 대응하는 실습 코드 부품과 부재 시 발생하는 실패 정의
layers = [("Prompt", contract_prompt, "역할 이름만 받은 Worker가 다음 단계에서 파싱할 수 없는 자유 형식 출력을 냅니다"),
          ("Context", WorkerResult, "한 줄 요약만 전달되어 세부 조건과 근거(Evidence)가 사라집니다"),
          ("Context", trim_payload, "단순 바이트 자르기로 인해 단서·예외 조항 문장이 잘려 나갑니다"),
          ("Harness", Budget, "모델 호출이 끝없이 늘어나도 실행을 멈추지 못합니다"),
          ("Harness", allowlist_middleware, "Worker가 자신의 역할에 허용되지 않은 도구를 호출합니다"),
          ("Harness", check_refund, "모델이 제안한 환불이 아무런 검증 없이 즉시 실행됩니다"),
          ("Graph", join_results, "실패한 Worker가 합류 단계에서 조용히 무시됩니다"),
          ("Graph", build_hybrid, "실행 경로를 매번 모델이 임의로 새로 결정하게 됩니다"),
          ("Loop", next_action, "원인이 다른 모든 실패에 대해 무조건 똑같은 재호출만 반복합니다")]

show_table([{"층": layer, "부품": part.__name__, "파일과 줄": where(part), "없으면 생기는 실패": failure}
            for layer, part, failure in layers])
print("모델 호출 0회")

## 2. Prompt: 역할 이름과 작업 계약

**문제.** 프롬프트에 *"너는 장애 보상 전문가야"* 라고 한 줄 적어 주면 모델이 전문가처럼 빈틈없이 일할까요? 이러한 역할 부여 지시에는 **어떤 자료를 근거(Evidence)로 삼아야 하는지, 결과를 어떤 필드에 담아 반환할지, 자료가 부족할 때는 어떻게 표시할지, 무엇을 추측하면 안 되는지**에 대한 기준이 전혀 없습니다. 다음 단계의 노드(코드 또는 작성자 에이전트)는 Worker의 그럴듯한 말투가 아니라 **약속된 데이터 칸(필드)** 을 읽어야 합니다.

실습에서 `outage` Worker에게는 가용률 기록을 조회하는 도구만 부여합니다. 즉, 이 Worker는 고객의 기본 이용료도 알 수 없고 SLA 문서의 크레딧 산정 구간표도 조회할 수 없습니다. 이 상태에서 일부러 Worker의 권한 범위를 넘어서는 질문을 던져 보겠습니다: *"2026년 8월 가용률을 확인하고, 그에 따른 서비스 크레딧 금액을 알려 주세요."* 자신이 조회할 수 없는 값을 어떻게 처리하는지를 보면 단순 역할 프롬프트와 명시적 **작업 계약(Task Contract)** 의 차이가 선명하게 드러납니다.

**먼저 예측하세요.** 단순 역할 프롬프트만 받은 Worker의 자유 텍스트 출력을 다음 단계의 코드가 그대로 받아 처리할 수 있을까요? 실행하기 전에 다음 세 가지를 먼저 예측해 보세요:
1. 권한 밖인 크레딧 금액을 임의로 지어낼까요, 아니면 확인할 수 없다고 답할까요?
2. 근거로 삼은 조회 번호(`fact_ref`)를 답변에 빠짐없이 남길까요?
3. 다음 단계의 코드가 이 Worker의 작업 완료 여부(`status`)를 어느 값으로 판별할 수 있을까요?

아래 코드에서는 먼저 단순 역할 프롬프트만 부여한 Worker를 실행하여 그 출력을 확인합니다.

In [ ]:
# 1) 단순 역할 이름("너는 장애 보상 전문가야.")만 부여한 Worker를 실행합니다.
announce("역할 프롬프트 Worker를 실행합니다", model_calls=4)
GOAL = "2026년 8월 가용률을 확인하고, 그에 따른 서비스 크레딧 금액을 알려 주세요."
ROLE_PROMPT = "너는 장애 보상 전문가야."

role_desk, role_meter = make_desk(case, label="role"), ModelMeter()
role_agent = create_deep_agent(model=model, tools=role_desk.tools("outage"), system_prompt=ROLE_PROMPT, name="role-outage",
                               middleware=allowlist_middleware(role_desk.budget, ROLE_TOOLS["outage"]))
role_state = role_agent.invoke({"messages": [{"role": "user", "content": GOAL}]},
                               config={"callbacks": [role_meter], "recursion_limit": 40})
role_text = str(role_state["messages"][-1].text)

print("모델 호출:", role_meter.totals()["model_calls"], "| 도구 호출:", role_desk.budget.counts.get("tool", 0))
print(role_text)

In [ ]:
# 2) 작업·근거·출력·한계·권한을 명시한 작업 계약(contract_prompt)과 구조화된 출력(WorkerResult)으로 Worker를 실행합니다.
announce("작업 계약 Worker를 실행합니다", model_calls=4)
contract_task = WorkerTask(task_id="outage", goal=GOAL, allowed_tools=sorted(ROLE_TOOLS["outage"]))
contract_desk, contract_meter = make_desk(case, label="contract"), ModelMeter()

# Worker에게 실제로 전달되는 작업 계약 프롬프트 출력
print(contract_prompt(contract_task, contract_desk.scope))
print("-" * 40)

contract_result = run_worker(contract_task, contract_desk, model, config={"callbacks": [contract_meter]})
print("모델 호출:", contract_meter.totals()["model_calls"], "| 도구 호출:", contract_desk.budget.counts.get("tool", 0))
print(contract_result.model_dump_json(indent=2))

In [ ]:
# 두 Worker(역할 프롬프트 vs 작업 계약)의 출력을 다음 단계 코드 입장에서 비교합니다.
REF = re.compile(r"\bF-[\w-]+|\bE-[0-9a-f]{12}")
AMOUNT = re.compile(r"\d+\.\d+|\d[\d,]*%|\d{1,3}(?:,\d{3})+")

def amounts(text: str) -> set[str]:
    """텍스트에서 소수, 비율(%), 천 단위 쉼표 금액만 추출합니다(날짜 숫자는 제외)."""
    return {n.replace(",", "").removesuffix("%") for n in AMOUNT.findall(text)}

def source_of_numbers(text: str, desk: Desk) -> str:
    """출력에 등장한 숫자들 중 실제 도구 조회 원문에 존재하는 숫자의 비율을 확인합니다."""
    source = {n.replace(",", "") for n in re.findall(r"\d[\d,]*(?:\.\d+)?", desk.source_text())}
    found = amounts(text)
    return f"{len(found & source)}/{len(found)}개가 조회 값에 있음. 없는 값: {sorted(found - source) or '없음'}"

# 자유 텍스트 출력에는 status 필드가 없으므로 코드가 강제로 WorkerResult 형태로 감싸야 합니다.
role_as_result = WorkerResult(status="complete", facts=[role_text], evidence_refs=REF.findall(role_text))
contract_text = " ".join([*contract_result.facts, *contract_result.uncertainty])
known = {"역할 프롬프트": role_desk.known_refs(), "작업 계약": contract_desk.known_refs()}

show_table([{"지시": "역할 프롬프트", "출력 형태": "자유 텍스트", "status": "칸 없음", "검증 가능한 근거 수": len(set(role_as_result.evidence_refs) & known["역할 프롬프트"]),
             "불확실성": "칸 없음", "사실 수": "칸 없음", "숫자의 출처": source_of_numbers(role_text, role_desk),
             "validate_result": validate_result(role_as_result, known["역할 프롬프트"]) or "문제 없음"},
            {"지시": "작업 계약", "출력 형태": "WorkerResult", "status": contract_result.status, "검증 가능한 근거 수": len(set(contract_result.evidence_refs) & known["작업 계약"]),
             "불확실성": len(contract_result.uncertainty), "사실 수": len(contract_result.facts), "숫자의 출처": source_of_numbers(contract_text, contract_desk),
             "validate_result": validate_result(contract_result, known["작업 계약"]) or "문제 없음"}])
print("모델 호출 0회")

**관찰할 것.** 비교표에서 `출력 형태`, `status`, `불확실성` 열을 먼저 확인해 보세요. 사람이 읽기에는 역할 프롬프트 Worker의 자유 텍스트 답변도 충분히 훌륭해 보일 수 있습니다. 모르는 금액을 지어내지 않고 정직하게 한계를 밝혔거나 조회 번호(`F-AV-2026-08`)를 문장 속에 적었을 수도 있습니다. 하지만 **그 정직함은 줄글 문장 속에 묻혀 있습니다.** 다음 단계의 코드는 자연어 문장을 읽고 이 작업이 완료(`complete`)된 것인지 일부가 불확실(`partial`)한 것인지 스스로 구별할 수 없습니다. 반면 **작업 계약 Worker**는 확인한 사실과 확인하지 못한 사항을 각각 `facts`, `uncertainty`, `status`라는 명시된 칸에 분리하여 담습니다.

- `숫자의 출처` 열도 살펴보세요. 만약 조회 원문에 없는 숫자가 발견되었다면 위 출력 문장에서 그 숫자가 어떤 맥락으로 쓰였는지 직접 찾아보세요. 금액을 단정하지 않고 *"예를 들어 기본료가 100만 원이라면~"* 처럼 예시를 든 것일 수도 있습니다. 사람은 이를 예시로 이해하지만, 다음 단계의 코드는 실제 사실과 예시 숫자를 구별하지 못합니다. 만약 없는 값이 `'없음'`으로 나왔다면 이번 실행에서는 두 프롬프트 모두 없는 숫자를 지어내지 않은 것입니다. 그렇더라도 작업 계약의 가치가 줄어드는 것은 아닙니다. 작업 계약의 목적은 평균적인 문장 품질 향상이 아니라 **다음 단계가 신뢰하고 읽을 수 있는 데이터 구조를 보장하는 것**이기 때문입니다.
- `validate_result` 열도 확인해 보세요. 역할 프롬프트 행은 원래 `status` 칸이 없어 코드가 임의로 `complete`로 가정하고 정규식으로 참조 ID를 긁어 모은 결과입니다. 만약 작업 계약 행에서 `'등록되지 않은 근거'` 경고가 나타났다면, 계약을 부여받은 모델이라도 사건 번호(`INC-311`)처럼 공식 근거 참조(`fact_ref`/`evidence_id`)가 아닌 값을 `evidence_refs` 칸에 넣을 수 있다는 뜻입니다. 프롬프트에 적은 계약은 물리적 강제가 아니라 '지시'일 뿐이므로, 이어지는 3절에서 코드 검증을 결합해야 합니다.
- 역할 프롬프트 자체가 무의미하다는 뜻은 아닙니다. 역할 부여는 답변의 어조와 도메인 관점을 잡는 데 유용하지만, 역할 이름만으로 멀티에이전트 간 인터페이스 계약이 완성되지는 않습니다. 또한 이 비교는 1회 실행 관찰이므로, 점수가 아닌 **구조적 차이(어느 칸이 존재하고 검증 가능한가)** 를 파악하는 데 집중해 주세요.

## 3. Context: 전달 계약과 손실

**문제.** Worker와 Worker 사이, 혹은 Worker와 최종 작성자(Writer) 사이에 **무엇을 어떤 형태로 넘기느냐**가 다음 단계가 볼 수 있는 정보의 세계를 결정합니다. 2절의 작업 계약이 개별 Worker에게 일을 시키는 약속이었다면, 3절의 **전달 계약**은 노드와 노드 사이에서 데이터를 주고받는 인터페이스 약속입니다. 입력 계약인 `WorkerTask`는 작업을 분배하는 노드가 Worker에게 전달하는 명세이고, 반환 계약인 `WorkerResult`는 Worker가 작업 수행 후 돌려주는 결과물입니다.

아래 코드가 출력하는 스키마 표를 보며 왜 입력 계약(`WorkerTask`)에 `allowed_tools` 필드가 포함되어 있는지 생각해 보세요. 각 작업이 사용할 수 있는 도구 권한을 작업 명세 자체에 적어 두면, 실행하기 전에 권한이 적절한지 사전 검토할 수 있고, 결과를 받은 뒤에도 *"이 Worker가 자신의 권한 밖인 사실을 어떻게 알았는가?"* 를 추적할 수 있습니다. 마찬가지로 반환 계약(`WorkerResult`)이 단순 요약문 대신 `facts`, `evidence_refs`, `uncertainty`를 별도 필드로 분리해 두는 이유는 다음 단계의 노드가 사실과 근거(Evidence)를 1:1로 대조하고 확인되지 않은 불확실성을 그대로 물려받도록 하기 위해서입니다.

아래 코드는 `WorkerTask`와 `WorkerResult` 두 계약 스키마의 필드 구성을 표로 출력하고, 이어서 잘못된 반환 결과 4종을 스키마 검사와 근거 검증에 차례로 통과시켜 봅니다.

In [ ]:
# WorkerTask(입력 계약)와 WorkerResult(반환 계약)의 Pydantic 스키마 구조를 표로 확인합니다.
def schema_rows(contract) -> list[dict]:
    """Pydantic 모델의 JSON 스키마에서 필드별 타입, 허용 값, 필수 여부, 기본값을 추출합니다."""
    schema = contract.model_json_schema()
    return [{"계약": contract.__name__, "필드": name, "타입": spec.get("type", "object"), "허용 값": spec.get("enum", ""),
             "필수": name in schema.get("required", []), "기본값": spec.get("default", "")}
            for name, spec in schema["properties"].items()]

show_table(schema_rows(WorkerTask) + schema_rows(WorkerResult))
print("allowed_tools를 작업 계약에 명시해 두면 해당 작업의 권한 범위를 실행 전에 검토할 수 있습니다.")
print("모델 호출 0회")

In [ ]:
# 실패 재현: 잘못된 Worker 결과 네 개를 스키마와 근거 검증에 차례로 넣습니다.
check_desk = make_desk(case, label="check")
check_desk.tools("outage")[0].invoke({"month": "2026-08"})  # 실제 도구 조회를 1회 실행하여 F-AV-2026-08을 작업대 근거로 등록합니다.
refs = check_desk.known_refs()

# 4가지 유형의 잘못된 WorkerResult 데이터 준비
bad_results = {"없는 status": {"status": "done", "facts": ["가용률 99.3"], "evidence_refs": ["F-AV-2026-08"]},
               "없는 근거 ID": {"status": "complete", "facts": ["가용률 99.3"], "evidence_refs": ["lookup_outage: 8월 장애"]},
               "근거 없는 complete": {"status": "complete", "facts": ["가용률 99.3"], "evidence_refs": []},
               "실제 근거에 틀린 사실": {"status": "complete", "facts": ["가용률 99.95"], "evidence_refs": ["F-AV-2026-08"]}}

rows = []
for name, data in bad_results.items():
    try:
        result, schema = WorkerResult(**data), "통과"
    except ValidationError as error:
        result, schema = None, f"거절: {error.errors()[0]['type']}"
    problems = validate_result(result, refs) if result else None
    rows.append({"잘못된 결과": name, "Pydantic 스키마": schema, "validate_result": "검사 전에 멈춤" if result is None else problems or "통과",
                 "거절한 층": "스키마" if result is None else "근거 검증" if problems else "어느 층도 거절하지 않음"})

show_table(rows)
print("등록된 근거:", sorted(refs), "| 모델 호출 0회")

위 표의 첫 세 행은 서로 다른 검증 층에서 걸러집니다. **Pydantic 스키마**는 데이터의 외형(타입과 허용된 열거형 값)을 검사하고, **`validate_result` 함수**는 인용된 참조 ID가 실제로 이번 실행의 작업대(`Desk`)에 등록된 근거(Evidence)인지를 검사합니다. 특히 두 번째 행의 `"lookup_outage: 8월 장애"`는 임의로 지어낸 예시가 아니라, 실제 테스트 실행 중 모델이 `evidence_refs` 리스트에 넣었던 값입니다. 자료형 자체는 문자열 리스트(`list[str]`)이므로 Pydantic 스키마는 그대로 통과하지만, 작업대의 실제 근거 목록(`refs`)과 대조하는 `validate_result` 층에서 차단됩니다.

특히 **네 번째 행(`실제 근거에 틀린 사실`)** 을 주의 깊게 살펴보세요. 인용한 근거 ID(`F-AV-2026-08`)는 실제 작업대에 등록된 진짜 ID이지만, 정작 `facts`에 적힌 수치(`99.95`)는 원문(`99.3`)과 다릅니다. 스키마 검사와 근거 ID 검증을 두 겹으로 두었음에도 이 네 번째 행은 두 검사를 모두 통과합니다. 코드 기반의 근거 ID 검증은 *"해당 근거를 실제로 조회했는가"* 만 확인할 뿐, *"요약한 문장의 의미가 원문과 일치하는가"* 까지 판별하지는 못하기 때문입니다. 따라서 유효한 인용 ID가 붙어 있더라도 중요한 수치는 원문(`quote`)과 대조하거나 코드 계산기로 검증하는 장치가 추가로 필요합니다.

이제 Worker의 결과를 작성자(Writer)에게 넘기는 **전달 형태(Payload Format)** 의 차이를 실험해 보겠습니다. 현업에서 멀티에이전트를 설계할 때 흔히 *"컨텍스트가 너무 길어지니 각 에이전트가 핵심만 1줄로 요약해서 넘기자"* 라는 아이디어를 떠올립니다. 요약은 분명 입력 토큰을 줄여 주지만, 정작 **무엇이 함께 지워지는지**는 알려 주지 않습니다. 이 아이디어를 관통 사례 `T-1001`에서 직접 측정해 보겠습니다.

**먼저 예측하세요.** `T-1001`에서 서비스 크레딧은 '기본 이용료'만을 기준으로 계산해야 하고(`sla-a-v2.md:11`), 신청 기한은 본문(15일)이 아니라 추가 약정(`sla-a-amendment.md:6`)의 **익월 20일**을 따라야 합니다. 만약 각 Worker가 가장 첫 번째 사실 **1줄만** 작성자에게 넘긴다면 어떤 단서 조항이 먼저 사라질까요? 작성자가 총 결제 금액(1,200,000원)을 계산 기준으로 잘못 사용할까요, 아니면 계산 기준을 알 수 없다며 금액 산정을 보류할까요? 신청 기한은 15일과 20일 중 무엇으로 답할까요? 먼저 예측한 뒤 아래 셀들을 실행해 보세요.

### 그림으로 보기: 잘못된 결과를 어느 층이 거절하는가

위 표의 `거절한 층` 열을 다이어그램으로 나타내면 다음과 같습니다. 네 가지 잘못된 결과가 왼쪽에서 오른쪽으로 두 단계의 검증 필터를 차례로 통과합니다.

![잘못된 Worker 결과 네 개가 Pydantic 스키마와 validate_result를 차례로 지난다. 없는 status는 스키마에서, 없는 근거 ID와 근거 없는 complete는 validate_result에서 거절되고, 실제 근거에 틀린 사실은 어느 검사도 거절하지 않고 통과한다](./assets/04a-1-check-layers.png)

- **1차 필터(Pydantic 스키마)**: 데이터의 모양과 타입을 검사합니다. `status`가 허용된 값(`complete`, `partial`, `failed`)이 아니면 이 단계에서 즉시 거절됩니다.
- **2차 필터(`validate_result`)**: 이번 실행의 작업대에 실제로 등록된 근거(Evidence)인지를 대조합니다. 문자열 타입이라 스키마를 통과한 가짜 근거 ID나 근거 없는 완료 주장을 여기서 잡아냅니다.
- **통과 후 남는 위험**: 마지막 네 번째 사례처럼 진짜 근거 ID를 인용하면서 사실 수치를 잘못 적은 경우에는 두 코드 검사를 모두 통과하므로, 원문(quote) 전달이나 코드 계산 도구가 뒷받침되어야 합니다.

아래 코드에서는 먼저 `T-1001` 사례에 대해 3개 역할(`billing`, `outage`, `policy`)의 Worker를 병렬로 1회 실행하여 기준 결과를 확보합니다.

In [ ]:
# T-1001에 대해 3명의 Worker를 한 번만 실행하고, 그 결과를 고정하여 3가지 전달 형태를 비교합니다.
announce("T-1001 Worker 3개를 병렬로 실행합니다", model_calls=12)
case_desk, case_meter = make_desk(case, label="payloads"), ModelMeter()

def run_role(role: str) -> tuple[str, WorkerResult]:
    """각 역할(billing, outage, policy)에 맞는 WorkerTask를 생성하여 실행합니다."""
    task = WorkerTask(task_id=role, goal=f"{ROLE_GOALS[role]}\n고객 요청: {case['question']}", allowed_tools=sorted(ROLE_TOOLS[role]))
    return role, run_worker(task, case_desk, model, config={"callbacks": [case_meter]})

with ThreadPoolExecutor(max_workers=3) as pool:
    worker_results = dict(pool.map(run_role, ROLES))

# 뒤 셀들은 이 결과를 고정된 값으로 읽습니다(예산의 시간 한도가 뒤 셀에 영향을 주지 않도록 값을 별도 저장합니다).
case_refs, case_evidence = case_desk.known_refs(), dict(case_desk.context.evidence)
case_quotes = {**dict(case_desk.fact_refs), **{ref: item.quote for ref, item in case_evidence.items()}}
results_hash = plan_hash({role: result.model_dump() for role, result in worker_results.items()})

show_table([{"작업": role, "status": r.status, "사실 수": len(r.facts), "근거 수": len(r.evidence_refs), "불확실성 수": len(r.uncertainty),
             "검증 문제": validate_result(r, case_refs) or "없음", "오류": r.error} for role, r in worker_results.items()])
print(timeline(case_desk.trace.spans))
print("모델 호출:", case_meter.totals()["model_calls"], "| Worker 결과 지문:", results_hash)

In [ ]:
# 방금 얻은 동일한 Worker 결과(worker_results)로부터 3가지 전달물(1줄 요약 / 사실+근거+불확실성 / 원문 포함)을 구성합니다.
def quotes_of(result: WorkerResult) -> dict[str, str]:
    """WorkerResult가 인용한 근거 ID들에 해당하는 실제 원문(quote) 딕셔너리를 반환합니다."""
    return {ref: case_quotes[ref] for ref in result.evidence_refs if ref in case_quotes}

def summary_line(result: WorkerResult) -> str:
    """요약 과정에서 다른 모델 변수가 개입하지 않도록 코드로 첫 번째 사실 1줄만 추출합니다."""
    return result.facts[0] if result.facts else f"(사실 없음: {result.status})"

# 세 가지 전달 형태 구성 및 바이트 크기·근거 수 비교
evidence_shape = {role: r.model_dump(include={"status", "facts", "evidence_refs", "uncertainty", "error"}) for role, r in worker_results.items()}
payloads = {"1줄 요약": {role: summary_line(r) for role, r in worker_results.items()},
            "사실+근거+불확실성": evidence_shape,
            "원문 포함": {role: {**evidence_shape[role], "quotes": quotes_of(r)} for role, r in worker_results.items()}}
payload_text = {name: json.dumps(p, ensure_ascii=False, indent=1) for name, p in payloads.items()}

show_table([{"전달물": name, "바이트": len(text.encode("utf-8")), "인용 근거 참조": len(set(REF.findall(text))),
             "불확실성 칸": "uncertainty" in text, "Worker 결과 지문": results_hash} for name, text in payload_text.items()])
print(payload_text["1줄 요약"])
print("모델 호출 0회 (전달 형태 하나만 독립 변수로 비교하기 위해 Worker를 다시 실행하지 않습니다)")

In [ ]:
# 동일한 작성자(Writer) 프롬프트에 세 가지 전달물을 각각 넣어 최종 답변이 어떻게 달라지는지 확인합니다.
announce("같은 작성자에게 전달물 3종을 차례로 전달합니다", model_calls=3)
WRITER_PROMPT = ("고객에게 보낼 답을 씁니다. {scope}\n{guide}\n"
                 "환불이 맞다면 proposal에 action, payment_id, amount, reason을 쓰세요. 실행 여부는 코드 게이트가 정합니다.\n"
                 "아래 전달물이 당신이 가진 자료의 전부입니다. 보상 금액은 전달물로 계산할 수 있으면 계산하고, 확인되지 않는 것은 uncertainty에 쓰세요.\n"
                 "evidence_refs에는 전달물에 적힌 fact_ref와 evidence_id만 넣으세요.\n고객 요청: {question}\n전달물:\n{payload}")
guide = "\n".join(OUTCOME_GUIDE.splitlines()[:3])
writer, writer_meter, writer_rows = model.with_structured_output(FinalAnswer, method="json_schema"), ModelMeter(), []

for name, text in payload_text.items():
    prompt = WRITER_PROMPT.format(scope=scope_line(case_desk), guide=guide, question=case["question"], payload=text)
    try:
        final, error = writer.invoke(prompt, config={"callbacks": [writer_meter]}), ""
    except Exception as failure:  # 호출 실패 시에도 누락하지 않고 표에 행으로 기록합니다.
        final, error = None, type(failure).__name__
    writer_rows.append({"전달물": name, "final": final, "오류": error, "입력 토큰": writer_meter.calls[-1]["input_tokens"], "지문": results_hash})
    print(f"--- {name}\n{final.text if final else error}\n불확실성: {final.uncertainty if final else ''}")

print("모델 호출:", writer_meter.totals()["model_calls"], "| usage 누락:", writer_meter.totals()["usage_missing"])

In [ ]:
# 세 가지 전달물로 생성된 작성자의 최종 답변을 자동 검사표와 세부 조건 포함 여부 표로 비교합니다.
def writer_row(item: dict) -> dict:
    """작성자 응답을 evaluate_row가 검사할 수 있는 표준 행 딕셔너리로 변환합니다."""
    final = item["final"]
    refs = final.evidence_refs if final else []
    return {"structure": item["전달물"], "case_id": case["id"], "answer": final.text if final else "", "uncertainty": final.uncertainty if final else [],
            "outcome": final.outcome if final else None, "cited_docs": sorted({case_evidence[r].doc_id for r in refs if r in case_evidence}),
            "ledger_refunds": 0, "status": "ok" if final else "model_error"}

# 계산 기준 조건(사용량 요금 제외)과 신청 기한(20일 vs 15일)이 전달물과 답변에 각각 들어 있는지 정규식으로 점검
CONDITION = re.compile(r"계산 기준|포함하지 않|(?:사용량|사용료).{0,30}제외")
DEADLINE_20, DEADLINE_15 = re.compile(r"(?<!\d)20일|09-20"), re.compile(r"(?<!\d)15일|09-15")
checked, condition_rows = [], []

for item in writer_rows:
    row, final = writer_row(item), item["final"]
    check = evaluate_row(row, gold)
    refs = final.evidence_refs if final else []
    answer, given = row["answer"] + " ".join(row["uncertainty"]), payload_text[item["전달물"]]
    checked.append({"전달물": item["전달물"], "입력 토큰": item["입력 토큰"], "자동 검사": check["자동 검사"], "outcome": check["실제 outcome"], "누락 사실": check["누락 사실"],
                    "금지 포함": check["금지 포함"], "누락 문서": check["누락 문서"], "유효 인용": sum(r in case_refs for r in refs), "없는 인용": [r for r in refs if r not in case_refs]})
    condition_rows.append({"전달물": item["전달물"], "전달물에 계산 기준 조건": bool(CONDITION.search(given)), "답에 계산 기준 조건": bool(CONDITION.search(answer)), "답에 150,000": "150,000" in answer or "150000" in answer,
                           "전달물에 20일 기한": bool(DEADLINE_20.search(given)), "답에 20일 기한": bool(DEADLINE_20.search(answer)), "답에 15일 기한": bool(DEADLINE_15.search(answer))})

show_table(checked)
show_table(condition_rows)
print("자동 검사는 문자열 포함 여부만 보므로, 표현 차이로 인한 오판이 없는지 위 출력 답안과 함께 확인하세요. / 모델 호출 0회")

In [ ]:
# 컨텍스트 예산(바이트 상한)이 줄어들 때 trim_payload가 어떤 항목부터 순서대로 덜어내는지 확인합니다.
policy_result = worker_results["policy"]
policy_quotes = quotes_of(policy_result)
full_size = trim_payload(policy_result, policy_quotes, 10**9)["bytes"]

rows = []
for limit in (full_size, int(full_size * 0.6), int(full_size * 0.3), 120):
    trimmed = trim_payload(policy_result, policy_quotes, limit)
    kept = trimmed["payload"]
    rows.append({"예산 바이트": limit, "남은 바이트": trimmed["bytes"], "예산 초과": trimmed["over_budget"], "뺀 항목 수": len(trimmed["dropped"]),
                 "뺀 항목": trimmed["dropped"][:3], "남은 사실": len(kept["facts"]), "남은 원문": len(kept["quotes"]),
                 "남은 근거 참조": len(kept["evidence_refs"]), "불확실성 유지": kept["uncertainty"] == policy_result.uncertainty})

show_table(rows)
print("덜어내는 순서는 1순위 원문(quotes), 2순위 사실(facts)이며, 근거 참조(evidence_refs)와 불확실성(uncertainty)은 끝까지 보존합니다. / 모델 호출 0회")

In [ ]:
# 실패 재현: 바이트에서 그냥 자르면 예외 문장이 사라집니다.
cut_desk = make_desk(case, label="cut")
clause = cut_desk.context.read_more("sla-a-v2", 12, 3)["evidence"]["quote"]  # sla-a-v2.md 12~14행의 제외 시간 조항 원문
EXCEPTION = "단, 월 누적 120분을 초과한 점검 시간은 포함한다"
raw = clause.encode("utf-8")

rows = []
for limit in (len(raw), 180, 120, 100, 60):
    kept = raw[:limit].decode("utf-8", errors="ignore")
    rows.append({"예산 바이트": limit, "예외 문장 남음": EXCEPTION in kept, "남은 끝부분": kept[-34:].replace("\n", " ")})
show_table(rows)

# 검색 단계에서 애초에 빠진 문서는 뒤에서 아무리 계약을 잘 짜도 복원할 수 없습니다.
query = "서비스 크레딧 신청 기한"
shown = [{"검색 limit": limit, "찾은 문서": sorted({e["doc_id"] for e in search["evidence"]}), "찾지 못한 관련 문서": search["related_docs"]}
         for limit in (1, 6) for search in [make_desk(case, label="search").context.search(query, limit=limit)]]
show_table(shown)
print("모델 호출 0회")

**관찰할 것.** 위 실험 결과에서 두 가지 크기를 구분하여 읽어 보세요. `입력 토큰`은 작성자 모델 호출 전체에 들어간 토큰 수이고, 전달물의 `바이트`는 그중 Worker가 넘긴 근거 Context의 크기입니다. 작성자 지시문과 출력 스키마는 세 호출 모두 동일하므로, 입력 토큰의 차이는 순전히 전달물의 크기에서 비롯됩니다. 만약 입력 토큰이 줄어들면서 `누락 사실`이나 `누락 문서`가 늘어났다면, **반드시 토큰 절감량과 정보 누락을 함께 보고해야 합니다.** 토큰이 줄었다는 절반의 사실만 보고하면 품질 손실을 비용 절감 이득으로 오판하게 됩니다.

- 첫 번째 `자동 검사` 표는 **1줄 요약** 행부터 확인해 보세요. 만약 `금지 포함`에 `180000`이 있다면 '사용량 초과 요금 제외'라는 계산 기준 단서가 요약 과정에서 사라져 총액(1,200,000원)의 15%를 잘못 계산한 것입니다. 반대로 `누락 사실`에 `150000`이 있고 금지 표현이 없다면, 작성자가 단서 부족을 인지하고 금액 확정을 정직하게 보류한 것입니다. 또한 답변 문장이 그럴듯해 보여도 `유효 인용`이 `0`이면 다음 단계에서 대조할 근거(Evidence)가 전혀 남지 않습니다. `없는 인용` 열에 값이 있다면 무엇을 적었는지 확인해 보세요. 1줄 요약에는 근거 ID가 하나도 없기 때문에 모델이 전달물의 딕셔너리 키 이름(`billing`)이나 사건 번호(`INC-311`)를 근거 ID처럼 적어 넣기도 합니다.
- 두 번째 세부 조건 표에서 **`전달물에 ~`으로 시작하는 열이 `False`이면 작성자에게 넘기기 전에 이미 요약 과정에서 조건을 잃어버린 것**이고, **`전달물에 ~`는 `True`인데 `답에 ~` 열이 `False`이면 정보는 전달되었으나 작성자 모델이 답변에 반영하지 않은 것**입니다. 두 경우의 원인이 다르므로 수정해야 할 위치(Context 전달 방식 vs 작성자 Prompt/도구)도 달라집니다.
- 두 표의 판정이 엇갈리는 행도 찾아보세요. 예를 들어 첫 표의 `누락 사실`에는 `9월20일`이 떠 있는데 두 번째 표의 `답에 20일 기한`은 `True`라면, 작성자가 `2026-09-20`이나 `익월 20일`처럼 다른 표기법으로 맞게 작성한 경우입니다. 단순 문자열 포함 검사(`evaluate_row`)가 표기 차이를 놓친 것이므로, 이 열의 이름이 `정답`이 아니라 `자동 검사`인 이유를 여기서 확인할 수 있습니다.
- `사실+근거+불확실성` 행과 `원문 포함` 행의 결과를 비교해 보세요. 만약 표기 차이를 제외하고 두 행의 품질이 동일하다면, 이번 1회 실행에서는 원문(`quotes`)을 추가로 넘긴 토큰 비용만큼 필수 사실이 더 늘어나지는 않은 것입니다. 그렇다고 해서 원문 전달이 항상 불필요한 것은 아닙니다. Worker가 원문의 수치를 잘못 요약했을 때 작성자가 오류를 바로잡을 수 있는 유일한 자료가 원문이기 때문입니다.
- 참고로 현재 작성자에게는 계산기 도구가 없습니다. 숫자 계산을 LLM의 암산에 맡기면 환각 위험이 커지므로, 곱셈은 코드 도구(`calculate_credit`)에 맡기고 모델은 적용할 비율과 기준 금액만 선택하게 하는 방법을 다음 `04b`의 8절에서 다룹니다.
- `trim_payload` 표에서는 예산 바이트가 줄어들 때 **`남은 원문`이 가장 먼저 줄어들고, 그다음에 `남은 사실`이 줄어들며, `남은 근거 참조`와 `불확실성 유지`는 끝까지 지켜지는지** 확인하세요(단, 앞선 Worker 실행에서 `policy` Worker가 `evidence_refs`를 0개 반환했다면 처음부터 줄일 원문과 근거 참조가 없어 `남은 사실`만 줄어들게 됩니다). 가장 작은 예산(120바이트)에서 `예산 초과`가 `True`로 표시되는 것은, 최소한으로 남겨야 할 근거 참조와 불확실성 자체의 크기가 120바이트보다 크다는 뜻입니다. 즉, 핵심 안전장치를 임의로 잘라내지 않고 예산 초과 사실을 투명하게 보고하도록 설계되어 있습니다.
- 마지막으로 **단순 바이트 자르기(`raw[:limit]`)** 표와 **검색 `limit` 비교** 표를 확인해 보세요. 문장을 바이트 길이로 뚝 자르면 `sla-a-v2.md:13`의 핵심 단서(*"단, 월 누적 120분을 초과한 점검 시간은 포함한다"*)가 흔적도 없이 사라지지만, 남은 문장은 문법적으로 멀쩡해 보여서 아무런 에러가 나지 않습니다. 또한 검색 단계에서 `limit=1`로 설정해 추가 약정 문서(`sla-a-amendment`)가 애초에 수집되지 않았다면, 뒤에서 아무리 전달 계약을 정교하게 짜도 잃어버린 약정을 되살릴 수 없습니다. 이 손실은 Context 전달이 아니라 검색(Retrieval) 설계로 돌아가서 해결해야 합니다.

### 그림으로 보기: 전달물에 무엇이 들어가고 무엇이 먼저 빠지는가

앞서 비교한 세 가지 전달물의 구성 필드와 `trim_payload`의 축소 우선순위를 한 장의 그림으로 정리했습니다. 위쪽은 세 전달물에 포함되는 필드이고, 아래쪽은 바이트 예산이 부족할 때 덜어내는 순서입니다.

![세 전달물에 들어가는 칸. 1줄 요약은 첫 사실만, 사실+근거+불확실성은 status, facts, evidence_refs, uncertainty, error를, 원문 포함은 거기에 quotes를 더한다. trim_payload는 quotes, facts 순으로 빼고 evidence_refs와 uncertainty는 끝까지 남긴다](./assets/04a-2-payload-fields.png)

- **1줄 요약의 구조적 한계**: `uncertainty`와 `evidence_refs` 칸 자체가 존재하지 않으므로, 하위 Worker가 발견한 불확실성이나 근거 참조를 다음 단계로 물려줄 수 없습니다.
- **`trim_payload`의 보존 원칙**: 컨텍스트 크기를 줄이더라도 코드가 정한 우선순위(원문 제거 → 사실 제거)에 따라 항목 단위로 덜어내고, **근거 참조(`evidence_refs`)와 불확실성(`uncertainty`)은 예산이 아무리 작아도 끝까지 남깁니다.**

### 그래프 안의 경계: 서브그래프가 주고받는 칸

지금까지의 전달 계약은 Worker와 작성자 사이에서 **무엇을 어떤 모양으로** 넘기는지였습니다. 그래프를 여러 층으로 쌓으면(02 5절의 Hierarchical처럼 팀을 서브그래프로 만들어 상위 그래프의 노드로 붙이면) 계약이 하나 더 생깁니다. **상위 그래프와 서브그래프가 어느 State 칸을 함께 쓰는가**입니다.

LangGraph에서 컴파일한 서브그래프를 `add_node`로 그대로 붙이면, 두 그래프가 **이름이 같은 칸**으로만 값을 주고받습니다. 이름이 같은 칸은 서브그래프 입력으로 들어가고, 서브그래프가 끝나면 같은 이름의 칸으로 돌아옵니다. 이름이 다른 칸은 들어가지도 나오지도 않습니다.

**먼저 예측하세요.** 아래 실패 재현에서 상위 그래프는 질문을 `question` 칸에 두고 팀의 보고를 `team_report` 칸에서 기다립니다. 팀 서브그래프는 작업을 `task` 칸에서 읽고 보고를 `report` 칸에 씁니다. 두 그래프가 함께 쓰는 칸이 하나도 없습니다. 이 그래프를 컴파일하면 오류가 날까요? 실행하면 오류가 날까요? 오류가 나지 않는다면 상위 그래프의 `team_report`에는 무엇이 남을까요?

In [ ]:
# 실패 재현: 상위 그래프와 함께 쓰는 칸이 없는 서브그래프를 그대로 노드로 붙입니다
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


# 1단계: 팀 서브그래프. 작업을 task 칸에서 읽고, 안에서 긴 작업 기록(worklog)을 남기고, 보고를 report 칸에 씁니다.
class TeamState(TypedDict, total=False):
    task: str
    worklog: list[str]
    report: str

def team_research(state: TeamState) -> dict:
    """모델 대신 일반 함수로, 원문을 여러 건 읽고 보고를 쓰는 팀을 흉내 냅니다."""
    worklog = [f"원문 {number}번을 읽고 근거로 등록했습니다." for number in range(1, 6)]
    return {"worklog": worklog, "report": f"[팀 보고] 받은 작업: {state.get('task', '(받은 작업 없음)')} / 근거 {len(worklog)}건"}

team_builder = StateGraph(TeamState)
team_builder.add_node("research", team_research)
team_builder.add_edge(START, "research")
team_builder.add_edge("research", END)
team_graph = team_builder.compile()


# 2단계: 상위 그래프. 질문은 question 칸에 있고, 팀의 보고를 team_report 칸에서 기다립니다.
class ParentState(TypedDict, total=False):
    question: str
    team_report: str
    final: str

def write_final(state: ParentState) -> dict:
    return {"final": state.get("team_report") or "(받은 보고가 없어 빈 답을 씁니다)"}

def build_parent(team_node) -> object:
    builder = StateGraph(ParentState)
    builder.add_node("team", team_node)
    builder.add_node("writer", write_final)
    builder.add_edge(START, "team")
    builder.add_edge("team", "writer")
    builder.add_edge("writer", END)
    return builder.compile()


# 3단계: 컴파일한 서브그래프를 그대로 노드로 붙여 실행합니다. 함께 쓰는 칸이 없습니다.
shared = sorted(set(ParentState.__annotations__) & set(TeamState.__annotations__))
broken_parent = build_parent(team_graph)
broken_result = broken_parent.invoke({"question": case["question"]})
print("두 State가 함께 쓰는 칸:", shared or "없음")
print("컴파일과 실행: 예외 없음")
show_table([{"칸": key, "값": str(value)[:70]} for key, value in broken_result.items()])
print("모델 호출 0회")

In [ ]:
# 고친 판: 서브그래프가 내보낼 칸을 output_schema로 정하고, 상위 그래프와 주고받는 값을 감싸는 함수 하나에 적습니다.
class TeamOutput(TypedDict):
    report: str  # 팀 밖으로 나가는 칸은 보고 하나뿐입니다. worklog는 팀 안에 남습니다.

bounded_builder = StateGraph(TeamState, output_schema=TeamOutput)
bounded_builder.add_node("research", team_research)
bounded_builder.add_edge(START, "research")
bounded_builder.add_edge("research", END)
bounded_team = bounded_builder.compile()

def call_team(state: ParentState) -> dict:
    """전달 계약: 상위 → 팀으로 넘길 값(task)과 팀 → 상위로 돌려받을 값(team_report)을 여기서만 정합니다."""
    team_output = bounded_team.invoke({"task": state["question"]})
    return {"team_report": team_output["report"]}

fixed_parent = build_parent(call_team)
fixed_result = fixed_parent.invoke({"question": case["question"]})
direct = bounded_team.invoke({"task": case["question"]})  # 서브그래프를 따로 불러 무엇이 밖으로 나오는지도 봅니다.

show_table([
    {"방식": "그대로 붙임", "팀이 받은 작업": "넘긴 칸 없음" if not shared else ", ".join(shared), "상위의 team_report": broken_result.get("team_report", "(비어 있음)"),
     "최종 답": broken_result["final"][:60]},
    {"방식": "감싸는 함수 + output_schema", "팀이 받은 작업": "question을 task로 넘김", "상위의 team_report": fixed_result["team_report"][:60],
     "최종 답": fixed_result["final"][:60]},
])
print("output_schema를 단 팀이 밖으로 내보낸 칸:", sorted(direct), "| 팀 안의 worklog는 상위로 올라오지 않습니다")
print("모델 호출 0회")

**관찰할 것.** 실패 재현의 출력부터 읽으세요.

- `두 State가 함께 쓰는 칸`이 `없음`인데도 `컴파일과 실행: 예외 없음`이 찍혔습니다. LangGraph는 함께 쓰는 칸이 없는 서브그래프를 오류로 보지 않습니다. 이름이 같은 칸만 주고받는다는 규칙대로, 주고받을 것이 없으니 아무것도 주고받지 않았을 뿐입니다.
- 상위 그래프의 State 표에 `team_report`가 없고, `final`은 "받은 보고가 없어 빈 답"입니다. 팀은 안에서 원문 다섯 건을 읽고 보고를 썼지만 그 보고는 상위 그래프에 닿지 않았습니다. 팀은 `task` 칸도 받지 못했으니 무슨 작업인지도 모른 채 일했습니다. 3절 앞부분의 1줄 요약이 근거를 **줄여서** 잃었다면, 이것은 칸 이름 하나가 어긋나 **통째로** 잃은 것입니다. 그리고 이번에도 예외는 없습니다.
- 두 번째 표의 `감싸는 함수 + output_schema` 행에서는 질문이 `task`로 넘어가고 보고가 `team_report`로 돌아왔습니다. `call_team` 함수가 무엇을 넘기고 무엇을 돌려받는지를 한곳에 적은 **전달 계약**입니다. 칸 이름이 바뀌면 이 함수에서 바로 `KeyError`가 나므로, 손실이 조용히 지나가지 않고 예외로 드러납니다.
- 마지막 줄의 `밖으로 내보낸 칸`이 `['report']` 하나입니다. `output_schema`가 팀의 긴 작업 기록(`worklog`)을 팀 안에 가둡니다. 02 5절의 Hierarchical이 하위 팀의 전체 로그 대신 짧은 전달물만 위로 올린 것과 같은 경계를, 이번에는 그래프의 스키마로 정한 것입니다.

정리하면 그래프 안의 전달 계약은 두 겹입니다. 앞에서 본 Pydantic 스키마(`WorkerResult`)는 넘기는 값의 **모양**을 지키고, 서브그래프의 입출력(감싸는 함수와 `output_schema`)은 값이 **어느 칸으로 들고 나는지**를 지킵니다. 이 셀의 팀은 모델 없이 일반 함수로 만들었으므로 실행할 때마다 같은 결과가 나옵니다.

## 4. Harness: 공유 예산

**문제.** 병렬로 실행되는 Worker 세 명이 각자 자기만의 호출 한도(예: 각 5회)를 따로 가지고 있다면, 시스템 전체의 총비용은 누가 통제할 수 있을까요? 재시도 루프와 병렬 분기가 결합되면 전체 모델 호출 수는 순식간에 불어납니다. 모델 스스로는 자신이 지금까지 총 몇 번 호출되었는지 알지 못하며, 설령 프롬프트로 알려 주더라도 *"예산을 넘으면 멈추라"* 는 자연어 지시는 물리적 강제력이 없습니다. **호출 집계, 검증, 강제 중단, 복구는 모델 바깥 시스템(Harness)의 책임입니다.**

`Budget` 클래스는 한 번의 요청 처리에서 발생하는 모든 모델·도구·검색 호출을 **단일 카운터**로 집계합니다. 먼저 모델을 호출하지 않고 멀티스레드 환경에서 공유 예산이 정확히 동작하는지 확인해 보겠습니다. 3개의 스레드가 각각 3번씩(총 9회) 소모를 시도하지만, 공유 예산의 한도(`max_model_calls`)는 `5`로 설정되어 있습니다. 아래 코드를 실행하여 여러 스레드가 동시에 접근하더라도 내부 락(Lock)을 통해 누적 허용 횟수가 정확히 한도(`5`)에서 멈추는지 확인합니다.

In [ ]:
# 3개의 스레드가 한도 5인 단일 Budget 객체에 동시에 소모를 요청할 때 정확히 5회만 허용되는지 검증합니다.
shared = Budget(max_model_calls=5)
log, log_lock = [], threading.Lock()

def consumer(name: str) -> None:
    """각 스레드가 3회씩 모델 호출 예산 소모를 시도하고 결과를 로그에 기록합니다."""
    for attempt in range(1, 4):
        time.sleep(0.01 * attempt)
        try:
            shared.consume("model", worker=name)
            outcome = "허용"
        except BudgetExceeded as error:
            outcome = f"BudgetExceeded: {error}"
        with log_lock:
            log.append({"순서": len(log) + 1, "스레드": name, "시도": attempt, "결과": outcome, "누적 model": shared.counts.get("model", 0)})

threads = [threading.Thread(target=consumer, args=(name,)) for name in ("billing", "policy", "outage")]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

show_table(log)
print("허용:", sum(r["결과"] == "허용" for r in log), "| 거절:", sum(r["결과"] != "허용" for r in log), "| 모델 호출 0회")

In [ ]:
# 실패 재현: 모델 호출 예산 2회로 Worker 셋을 차례로 실행합니다.
announce("예산 한도 2회로 Worker 실행", model_calls=2)
small_desk, small_meter = make_desk(case, budget={"max_model_calls": 2}, label="small"), ModelMeter()
rows, stopped = [], False

for role in ROLES:
    task = WorkerTask(task_id=role, goal=f"{ROLE_GOALS[role]}\n고객 요청: {case['question']}")
    if stopped:
        rows.append({"작업": role, "상태": "시작하지 않음", "오류": "앞 작업에서 예산 소진", "누적 model": small_desk.budget.counts.get("model", 0)})
        continue
    try:
        status, error = run_worker(task, small_desk, model, config={"callbacks": [small_meter]}).status, ""
    except BudgetExceeded as failure:
        status, error, stopped = "budget_exceeded", str(failure), True
    rows.append({"작업": role, "상태": status, "오류": error, "누적 model": small_desk.budget.counts.get("model", 0)})

show_table(rows)
report = small_desk.budget.report()
print("budget.report() counts:", report["counts"], "| 경과 초:", report["elapsed_seconds"], "| 실제 모델 호출:", small_meter.totals()["model_calls"])
show_table([{k: e.get(k, "") for k in ("event", "seconds", "name")} for e in report["events"] if e["event"] != "usage"])

**관찰할 것.** 첫 번째 스레드 실험 표에서 `누적 model` 값이 한도인 `5`를 넘어가는 행이 없는지 확인해 보세요. 스레드 스케줄링에 따라 어느 스레드가 먼저 거절당하는지는 실행할 때마다 달라질 수 있지만, 총 허용 건수는 언제나 한도(`5`)와 정확히 일치해야 합니다.

- 두 번째 표(실패 재현)에서는 각 작업의 `상태` 열을 확인합니다. 어느 Worker에서 공유 예산(`max_model_calls=2`)이 소진되었는지, 그리고 그보다 앞서 완료된 Worker의 결과는 살아 있는지 살펴보세요. `budget_exceeded`는 조용히 덮어두고 넘어갈 오류가 아니라, 상위 그래프가 명확히 인지하고 처리해야 할 실패 상태입니다.
- 세 번째 이벤트 표는 공유 예산이 어떤 모델·도구 호출들에 의해 몇 초 시점에 소진되었는지 시간 순서대로 투명하게 보여 줍니다.
- 예산 초과가 발생했을 때 가장 나쁜 대응은 **원인을 살피지 않고 무조건 예산 숫자(`max_model_calls`)만 늘려서 실패를 감추는 것**입니다. 한도를 올리기 전에 먼저 물어야 합니다: *"이 호출들이 정말 모두 필요했는가? 동일한 도구를 중복해서 호출하지는 않았는가? 특정 Worker 하나가 불필요한 재시도로 다른 Worker의 예산까지 독점하지는 않았는가?"* 이러한 분석을 거쳐 타당한 근거가 확인되었을 때만 예산을 조정하고 그 사유를 설계 기록에 남겨야 합니다.

### 그림으로 보기: 하나의 예산이 세는 방식

위에서 실행한 공유 예산 실험의 구조를 그림으로 나타내면 다음과 같습니다. 세 Worker 스레드는 각자 별도의 한도를 갖지 않고, 잠금(Lock)이 있는 단일 `Budget` 카운터에 소모를 요청합니다.

![billing, policy, outage 스레드가 잠금이 있는 Budget 카운터 하나에 소모를 요청한다. 한도 5 이하면 허용되고 한도를 넘으면 BudgetExceeded가 난다](./assets/04a-3-shared-budget.png)

- **단일 카운터의 필요성**: 호출 한도를 Worker마다 따로 부여하면 워크플로우 전체의 총비용 상한을 보장할 수 없습니다. 반드시 작업대(`Desk`)의 단일 `Budget` 인스턴스로 모아 집계해야 합니다.
- **결정적 상한 보장**: 병렬 실행 특성상 어느 Worker가 마지막 티켓을 가져가고 어느 Worker가 `BudgetExceeded`를 만날지는 실행마다 달라질 수 있지만, **전체 허용 건수가 한도를 절대 넘지 않는다**는 불변 조건은 항상 유지됩니다.

## 직접 해보기

두 실습 모두 한 가지 조건만 변경하여 전후 차이를 비교합니다. 실행하기 전에 무엇이 달라질지 먼저 예상해 본 뒤 실행해 보세요. **해보기 1**은 실제 모델을 호출하므로 `RUN_EXERCISE = True`로 변경해야 실행되며, **해보기 2**는 모델을 호출하지 않고 즉시 실행됩니다(실패 분기와 승인 게이트를 바꾸어 보는 실습은 다음 `04b`에 준비되어 있습니다).

### 해보기 1

2절에서 사용한 작업 계약(`TASK_CONTRACT`)에서 한 줄(`한계:` 또는 `권한:`)을 지우고 `outage` Worker를 다시 실행해 봅니다. 전체 계약을 모두 주었을 때와 비교해 어느 칸(`status`, `불확실성 수`, `숫자의 출처`)이 달라지는지 찾아보세요. 만약 1회 실행에서 결과가 크게 달라지지 않았다면, 단 1회 관찰만으로 프롬프트 규칙을 삭제해도 안전한지 함께 생각해 보세요. 아래 코드는 선택한 계약 문구를 제거한 뒤 Worker를 실행합니다.

In [ ]:
# 작업 계약에서 특정 줄(한계: 또는 권한:)을 제거했을 때 Worker 출력이 어떻게 달라지는지 실험합니다.
RUN_EXERCISE = False
REMOVE_LINE = "한계:"  # "권한:"으로 바꾸어 실험해 볼 수도 있습니다.
if RUN_EXERCISE:
    announce("해보기 1: 계약 한 줄 지우기", model_calls=4)
    contract = "\n".join(line for line in TASK_CONTRACT.splitlines() if not line.startswith(REMOVE_LINE))
    exercise_desk = make_desk(case, label="exercise-1")
    content = contract.format(goal=GOAL, entity=exercise_desk.scope.entity, as_of=exercise_desk.scope.as_of.isoformat(), facts="없음")
    state = make_worker("outage", exercise_desk, model).invoke({"messages": [{"role": "user", "content": content}]}, config={"recursion_limit": 40})
    answer = state["structured_response"]
    print(content)
    show_table([{"지시": name, "status": r.status, "사실 수": len(r.facts), "근거 수": len(r.evidence_refs), "불확실성 수": len(r.uncertainty),
                 "숫자의 출처": source_of_numbers(" ".join([*r.facts, *r.uncertainty]), d)}
                for name, r, d in (("전체 계약", contract_result, contract_desk), (f"{REMOVE_LINE} 삭제", answer, exercise_desk))])
else:
    print("실행하지 않았습니다. 값을 수정한 뒤 RUN_EXERCISE를 True로 바꾸어 실행해 보세요.")

### 해보기 2

3절의 표준 `trim_payload` 함수는 컨텍스트 예산이 부족할 때 원문(`quotes`)을 가장 먼저 빼고 사실(`facts`)을 그다음에 덜어내며, 불확실성(`uncertainty`)은 끝까지 남깁니다. 아래 코드에서 `DROP_ORDER` 리스트의 순서를 바꾸어 어떤 필드가 먼저 삭제되도록 할지 직접 지정해 보세요. 만약 `uncertainty`를 가장 먼저 지우도록 설정하면, 다음 단계의 작성자(Writer)는 무엇을 모른 채 답변을 작성하게 될까요? 그 선택이 고객에게 나갈 최종 답변을 어떻게 위험하게 만드는지 확인해 보세요.

In [ ]:
# 컨텍스트 축소 시 덜어낼 필드 순서(DROP_ORDER)를 바꾸어 표준 trim_payload와 비교합니다.
DROP_ORDER = ["uncertainty", "facts", "quotes"]  # 먼저 덜어낼 필드 이름부터 순서대로 적습니다.
MAX_BYTES = int(full_size * 0.5)

def my_trim(result: WorkerResult, quotes: dict[str, str], max_bytes: int, order: list[str]) -> dict:
    """지정한 필드 순서(order)대로 항목을 하나씩 제거하여 max_bytes 이하로 줄입니다."""
    payload = {"facts": list(result.facts), "evidence_refs": list(result.evidence_refs), "uncertainty": list(result.uncertainty), "quotes": dict(quotes)}
    size = lambda: len(json.dumps(payload, ensure_ascii=False).encode("utf-8"))
    dropped = []
    for field in order:
        while size() > max_bytes and payload[field]:
            item = payload[field].popitem()[0] if isinstance(payload[field], dict) else payload[field].pop()
            dropped.append(f"{field}: {item[:20]}")
    return {"payload": payload, "bytes": size(), "dropped": dropped}

mine, standard = my_trim(policy_result, policy_quotes, MAX_BYTES, DROP_ORDER), trim_payload(policy_result, policy_quotes, MAX_BYTES)
show_table([{"순서": name, "바이트": t["bytes"], "남은 사실": len(t["payload"]["facts"]), "남은 원문": len(t["payload"]["quotes"]),
             "남은 불확실성": len(t["payload"]["uncertainty"]), "뺀 항목 수": len(t["dropped"])}
            for name, t in (("내 순서", mine), ("trim_payload", standard))])
print("내 순서로 뺀 항목:", mine["dropped"], "| 모델 호출 0회")

## 정리

이번 `04a` 노트북에서 그래프에 추가한 엔지니어링 장치들을 다섯 층 지도 중 앞선 세 층(`Prompt`, `Context`, `Harness`)을 기준으로 정리합니다.

- **Prompt (작업 계약)**: 막연한 역할 이름이 아니라 **작업 목표, 근거 규칙, 출력 스키마, 한계, 허용 권한**을 명시한 계약으로 Worker를 정의했습니다. 작업 계약의 핵심 목적은 평균 문장 품질이 아니라 **다음 단계 코드가 신뢰하고 읽을 수 있는 필드를 보장하는 것**입니다.
- **Context (전달 계약과 축소 원칙)**: 사실(`facts`), 근거 참조(`evidence_refs`), 불확실성(`uncertainty`)을 별도 칸으로 분리해 전달했습니다. Pydantic 스키마 통과와 실제 등록 근거 검증(`validate_result`)은 서로 다른 층의 검사이며, 두 검사를 모두 통과하더라도 사실 내용이 원문과 일치한다는 보장은 없음을 확인했습니다. 서브그래프를 붙일 때는 감싸는 함수와 `output_schema`로 어느 칸이 들고 나는지까지 정해야, 칸 이름이 어긋나 결과가 통째로 사라지는 일을 예외로 드러낼 수 있습니다. 또한 컨텍스트를 줄일 때는 바이트 절단이 아니라 항목 단위로 덜어내고, 무엇을 뺐는지(`dropped`) 기록하며 근거 참조와 불확실성은 끝까지 보존합니다.
- **Harness (공유 예산)**: 병렬 Worker들의 호출 예산을 모델 외부의 단일 `Budget` 카운터로 집계하고, 한도 초과 시 즉시 중단하여 `budget_exceeded` 상태를 투명하게 드러냈습니다. 한도를 넘어 실패했을 때는 무작정 숫자를 올리기 전에 불필요한 중복 호출이 없었는지부터 점검해야 합니다.

남은 두 층인 **`Graph`** 와 **`Loop`** 는 이어지는 **`04b`** 노트북에서 다룹니다. 실패 유형에 따라 다음 행동을 세분화하고, 병렬 Worker의 부분 실패를 합류 노드에서 드러내며, 사람의 승인 게이트(HITL)와 원장 멱등성(Idempotency)을 그래프에 결합합니다.

**읽을거리.** 이 과정은 1절에서 정의한 자체 기준에 따라 Graph Engineering을 설명합니다. `"Graph Engineering"`이라는 용어는 2026년 무렵 업계에 확산된 외부 용어이며 글마다 포함하는 범위가 조금씩 다릅니다. 관심 있는 분들은 아래 두 외부 글을 비교용으로 읽어 보시길 권합니다. 어느 정의가 절대적으로 옳은지 따지기보다, 각 글이 무엇을 그래프의 책임으로 보는지 우리가 배운 다섯 층 지도에 대응시켜 읽어 보시면 좋습니다.

- https://towardsdatascience.com/graph-engineering-for-ai-agents-from-prompts-and-loops-to-workflows/
- https://rickhigh.substack.com/p/graph-engineering-is-two-ideas-wearing

## 확인 질문

1. 유효한 인용 ID(`evidence_refs`)가 들어 있고 Pydantic 스키마와 `validate_result`를 모두 통과한 `WorkerResult`라도 왜 100% 신뢰할 수 없나요? 3절의 잘못된 결과 비교표에서 어느 행이 그 근거인가요?
2. 3절의 전달물 3종 비교 실험에서 전달 형태의 순수한 효과를 측정하기 위해 Worker를 다시 실행하지 않고 고정한 이유는 무엇인가요? 만약 매번 Worker를 새로 실행했다면 어떤 변수와 뒤섞이게 되나요?
3. 함께 쓰는 칸이 없는 서브그래프를 노드로 붙였을 때 LangGraph가 예외를 내지 않은 이유는 무엇인가요? 그 손실을 예외로 드러나게 하려면 무엇을 바꾸어야 하나요?
4. Worker 세 명이 각자 개별 호출 한도를 갖는 방식과 단일 `Budget` 인스턴스를 공유하는 방식은 어떻게 다른가요? 예산 초과가 발생했을 때 한도 숫자를 늘리기 전에 먼저 무엇을 확인해야 하나요?

다음 **`04b`** 노트북에서는 오늘 구축한 계약과 공유 예산을 그대로 유지한 채 **실패 처리와 승인 제어**를 다룹니다. 실패 종류마다 다음 행동을 나누고, 병렬 Worker의 부분 실패를 합류에서 드러내고, 승인과 멱등을 그래프에 넣습니다.